# Retrieval-Augmented Generation (RAG)
## Information Retrieval and Search
### Keyword Search

In [2]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("gpreda/bbc-news")
news_data = pd.read_csv(os.path.join(path, "bbc_news.csv"))
news_data.head()

100%|██████████| 3.64M/3.64M [00:00<00:00, 120MB/s]

Extracting files...


,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [ ]:
import bm25s
corpus = list(news_data["title"] + " " + news_data["description"])
retriever = bm25s.BM25(corpus=corpus)

corpus_tokens = bm25s.tokenize(corpus)
print(f"Documents: {len(corpus_tokens.ids)}, Vocab: {len(corpus_tokens.vocab)}")
retriever.index(corpus_tokens)

query = "Retrieval augmented generation"
query_tokens = bm25s.tokenize(query)
docs, scores = retriever.retrieve(query_tokens, k=3)
print(f"Best result (score: {scores[0, 0]:.2f}): {docs[0, 0]}")

In [ ]:
# {24538: 'augmented', 3129: 'generation'}
id_query = {corpus_tokens.vocab[q]: q for q in list(query_tokens.vocab) if q in corpus_tokens.vocab}

# [19281, 17907, 18662]
id_doc = [corpus.index(d) for d in docs[0]]

count = pd.DataFrame({d: {id_query[t]: corpus_tokens.ids[d].count(t) for t in id_query} for d in id_doc}).T
score = pd.Series(scores[0], index=id_doc, name="score")

In [ ]:
count.join(news_data).join(score)

### Semantic Search

In [5]:
from sentence_transformers import SentenceTransformer
import joblib

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
data = list(news_data["title"] + " " + news_data["description"])
embeddings = model.encode(data)

joblib.dump(embeddings, model_name + ".joblib")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 581.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['all-MiniLM-L6-v2.joblib']

In [ ]:
embeddings = None
embeddings = joblib.load(model_name + ".joblib")